<a href="https://colab.research.google.com/github/haujla2391/CSCI-4170/blob/main/Lab03_HF_Pipelines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Checkpoint 1

In [ ]:
pip install -U transformers datasets evaluate accelerate torch torchvision pillow

In [ ]:
from transformers import pipeline
from datasets import load_dataset
import evaluate
import time
from PIL import Image

## Part A

## Task 1: Sentiment Analysis

In [ ]:
sentiment_analyzer = pipeline(
    task="text-classification",
    model='distilbert-base-uncased-finetuned-sst-2-english',
    device=0
)

sentiment_analyzer2 = pipeline(
    'text-classification',
    model='siebert/sentiment-roberta-large-english',
    device=0
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/256 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

## Task 2: Named Entity Recognition

In [ ]:
ner_tagger = pipeline(
    "ner",
    aggregation_strategy="simple",
    device=0
)

No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/998 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/60.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

# Part B

Sentiment dataset

In [ ]:
import random

dataset = load_dataset("imdb")

test_subset = dataset["test"].shuffle(seed=42).select(range(200))

texts = test_subset["text"]
labels = test_subset["label"]

accuracy = evaluate.load("accuracy")

def evaluate_model(pipeline_model, texts, labels):
    predictions = []

    start_time = time.time()

    for text in texts:
        result = pipeline_model(text[:512])[0]  # truncate long reviews
        label = 1 if result['label'] in ['POSITIVE', 'LABEL_1'] else 0
        predictions.append(label)

    total_time = time.time() - start_time
    avg_time = total_time / len(texts)

    acc = accuracy.compute(predictions=predictions, references=labels)

    return acc["accuracy"], avg_time

acc1, time1 = evaluate_model(sentiment_analyzer, texts, labels)
acc2, time2 = evaluate_model(sentiment_analyzer2, texts, labels)

print("Model 1 Accuracy:", acc1)
print("Model 1 Avg Time:", time1)

print("Model 2 Accuracy:", acc2)
print("Model 2 Avg Time:", time2)

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Model 1 Accuracy: 0.79
Model 1 Avg Time: 0.02590641736984253
Model 2 Accuracy: 0.855
Model 2 Avg Time: 0.011186357736587525


# Part C

# Version: 1.0.1 (Beta)


**Chosen task:** Sentiment Analysis (binary positive/negative classification on English text).

I compared the two models from the notebook on the same 200-sample IMDB test subset:

| Model Name/ID                              | Accuracy | Avg Inference Time (s/sample) |
|--------------------------------------------|----------|-------------------------------|
| distilbert-base-uncased-finetuned-sst-2-english | 0.79    | 0.0259                       |
| siebert/sentiment-roberta-large-english    | 0.855   | 0.0112                       |

**Short justification:**  
- `distilbert-base-uncased-finetuned-sst-2-english` is a DistilBERT checkpoint fine-tuned only on the Stanford Sentiment Treebank (SST-2) movie-review corpus. Intended use is straightforward text classification; limitations include potential bias toward underrepresented groups (e.g., country names in sentences trigger wildly different sentiment probabilities). License: Apache-2.0.  
- `siebert/sentiment-roberta-large-english` is a RoBERTa-large model fine-tuned on 15 diverse English datasets (Yelp reviews, Amazon product reviews, tweets, etc.) for broader generalization. Intended use is reliable binary sentiment analysis across varied text types. It explicitly notes outperforming the DistilBERT SST-2 model by >15 percentage points on average. There is no liscence listed.

**Surprising Behavior Observed:**  
It was surprising that the bigger model (`siebert/sentiment-roberta-large-english`, 1.42 GB) ran faster per sample than the smaller DistilBERT model (0.0111s for roberta vs 0.0259s for DistilBERT). This went against the usual expectation that larger models would be slower, yet here the more broadly trained RoBERTa delivered both better performance and lower latency on real IMDB reviews.